# CloudSRE v2 — Hackathon Training Notebook
This notebook trains a 1.5B agent using **GRPO** (Group Relative Policy Optimization) against a live 16-microservice architecture.

It uses a Dual-LLM architecture where the agent trains against our OpenEnv, while a 72B Adversarial Designer generates targeted faults.

# CloudSRE v2 — Final Training Pipeline (Hackathon Submission)

This notebook contains the complete reinforcement learning pipeline used to train our 1.5B parameter SRE agent to solve cascading failures. 

**The Pipeline:**
1. **SFT (Supervised Fine-Tuning):** We first warm up the base Qwen-1.5B model on a small dataset of Bash/SRE commands so it learns the *format* of an SRE (e.g., `systemctl restart`, `curl`, `cat`). This is the `cloudsre-1.5B-leg1` checkpoint.
2. **GRPO (Group Relative Policy Optimization):** This notebook loads that SFT checkpoint and trains it dynamically against our 16-node microservice simulation using TRL-style GRPO. It learns to actually *solve* the incidents through trial and error.

> **Note for Judges:** This notebook connects to our live HuggingFace Space environment. You can run it end-to-end to replicate our training. It will automatically generate `grpo_reward_curve.png` upon completion.


In [ ]:
# 1. Install Dependencies
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q httpx matplotlib wandb

In [ ]:
# 2. Clone the Environment Repository
!git clone https://github.com/Harikishanth/CloudSRE.git
%cd CloudSRE

In [ ]:
# 3. Authenticate with HuggingFace & WandB
from google.colab import userdata
import wandb

try:
    hf_token = userdata.get('HF_TOKEN')
    wandb_key = userdata.get('WANDB_API_KEY')
    
    !huggingface-cli login --token $hf_token
    wandb.login(key=wandb_key)
    print("✅ Successfully logged into HuggingFace and WandB")
except Exception as e:
    print("⚠️ Please add HF_TOKEN and WANDB_API_KEY to your Colab Secrets!")

## 4. Run the GRPO Training Loop
We are training the agent on `cascade` and `adversarial` scenarios. The script connects to the live HuggingFace Space environment.

In [ ]:
ENV_URL = "https://dardrax-cloudsre-environment.hf.space" # Update this if needed

!python train_grpo.py \
    --env-url {ENV_URL} \
    --model-id DarDrax/cloudsre-1.5B-leg1 \
    --curriculum warmup,single_fault,cascade,multi_cascade,adversarial \
    --episodes-per-tier 12 \
    --group-size 4 \
    --wandb-project CloudSRE-Hackathon-Run

## 5. Generate Professional Reward Graphs
Once training completes, this cell generates the professional PNG graphs for the README.

In [ ]:
# Assumes you downloaded the WandB CSV files to the root directory.
# If you haven't, you can just download the images directly from WandB.
import os
if os.path.exists("wandb_export_resolution.csv") and os.path.exists("wandb_export_reward.csv"):
    !python generate_plots.py \
        --resolution-csv wandb_export_resolution.csv \
        --reward-csv wandb_export_reward.csv
    from IPython.display import Image, display
    display(Image("cascade_resolution_rate.png"))
else:
    print("Upload your WandB CSVs here to generate plots locally, or just export the plots from the WandB dashboard directly.")